# Semantic Model Optimizer: run

Checks the semantic models in the workspaces you set below, each with the reports that use it, and adds the results to the default Lakehouse of this notebook. The tool itself is in the notebook **SemanticModelOptimizer-Tool**, in the same workspace; this one only holds your settings, so an update of the tool does not touch them.

**Before the first run:** import both notebooks into the same workspace, add a Lakehouse to this one (*Lakehouse* in the explorer › *Add*) and set the settings below. Then run all cells, and schedule this notebook (*Run* › *Schedule*). For another set of workspaces with its own schedule, make a copy of this notebook.

In [ ]:
# Which workspaces: a name, an id, several separated by commas, or "*" for every workspace you can open
WORKSPACES = "*"

# Folder under Files in the default Lakehouse
FOLDER = "SemanticModelOptimizer"

# Also write the results as tables (smo_...) in the Lakehouse, and create the report "Semantic Model Optimizer
# (Direct Lake)" on them in this workspace: no refresh to schedule and no sign-in to set up. False: only the files.
DIRECT_LAKE = True

# Azure Analysis Services (optional). Reports that are connected live to a model on Azure Analysis Services are
# checked with that model when you fill these in. A notebook cannot sign in to Analysis Services as you, so it needs
# an app registration (service principal) that is administrator of the server, with its client secret in a Key Vault
# that you can read.
AAS_TENANT = ""        # tenant id (Directory ID)
AAS_CLIENT_ID = ""     # application (client) id of the app registration
AAS_KEY_VAULT = ""     # address of the Key Vault, such as https://<name>.vault.azure.net/
AAS_SECRET_NAME = ""   # name of the secret in the Key Vault that holds the client secret

## The tool

Loads the notebook **SemanticModelOptimizer-Tool** from this workspace.

In [ ]:
%run SemanticModelOptimizer-Tool

## Check the models

In [ ]:
ctx = notebookutils.runtime.context
lakehouse_id = ctx.get("defaultLakehouseId")
if not lakehouse_id:
    raise Exception("This notebook has no default Lakehouse. Add one (Lakehouse in the explorer > Add) and run again.")
workspace_id = ctx.get("defaultLakehouseWorkspaceId") or ctx.get("currentWorkspaceId")

OUT = "/lakehouse/default/Files/" + FOLDER
# where the Power BI report reads the runs: the same folder, through OneLake
SOURCE = "https://onelake.dfs.fabric.microsoft.com/" + workspace_id + "/" + lakehouse_id + "/Files/" + FOLDER + "/data"

token = notebookutils.credentials.getToken("pbi")  # you, as you are signed in to Fabric
aas = None
if AAS_CLIENT_ID:  # the service principal for Azure Analysis Services; its secret stays in the Key Vault
    aas = aasToken(AAS_TENANT, AAS_CLIENT_ID, notebookutils.credentials.getSecret(AAS_KEY_VAULT, AAS_SECRET_NAME))
result = scan(WORKSPACES, OUT, token, aas_token=aas)
if not result["models"]:
    raise Exception("No model could be exported (see above).")

# the results as tables in the Lakehouse, and the Direct Lake report on them
if DIRECT_LAKE:
    writeLakehouseTables(spark, OUT)
    publishDirectLake(workspace_id, lakehouse_id, token)

## What this run found

In [ ]:
import pandas as pd
display(pd.DataFrame(result["models"]))

## The Power BI report

With `DIRECT_LAKE = True` the report *Semantic Model Optimizer (Direct Lake)* is in this workspace: open it, nothing to set up. It reads the `smo_` tables in the Lakehouse and shows each run as soon as the notebook is done.

Without it, or to have the report on your computer: the runs are also files in the Lakehouse. To see them in Power BI, do this once:

1. On your computer, with Semantic Model Optimizer (the setup or the portable exe from the releases), run the command the cell below prints. It writes the report, set up to read this Lakehouse.
2. Open *Semantic Model Optimizer.pbip* in Power BI Desktop, sign in with your organisational account when it asks for OneLake, and choose **Refresh**.
3. **Publish** it to a workspace. In the semantic model's settings, set the credentials of the data source (OAuth2) and schedule a refresh after the notebook's schedule.

In [ ]:
print("SemanticModelOptimizer-cli.exe cli --export C:/SMO-Fabric --report-only --report-source " + SOURCE)

## Done

When you run the notebook yourself, this stops the Spark session, so it does not stay open until Fabric ends it after 20 idle minutes with a timeout message. The results above stay. Scheduled runs and pipelines stop by themselves, so there it does nothing. To keep working in the session, skip this cell.

In [ ]:
ctx = notebookutils.runtime.context
if ctx.get("isForInteractive") and not ctx.get("isReferenceRun"):
    print("Done: the Spark session is stopped.")
    notebookutils.session.stop()